# 06 - Publication-Quality Visualization

Generates all figures needed for the paper:

1. **Figure 1**: Slice comparison (GT vs methods) with zoom regions
2. **Figure 2**: Error maps
3. **Figure 3**: Z-axis error profiles
4. **Figure 4**: Training convergence curves
5. **Figure 5**: PSNR/SSIM bar charts across methods
6. **Figure 6**: ROI (per-organ) comparison
7. **Figure 7**: Ablation bar chart

All figures saved at 300 DPI with proper font sizes for publication.

In [ ]:
# ========================== CONFIGURATION ==========================
FULL_EXPERIMENT = True
SEED = 42

# Cases to visualize (select representative ones)
VIZ_CASES = [0, 5, 10]  # Adjust based on available test cases
VIZ_SPARSE_RATIO = 3     # R=3 shows differences most clearly
DPI = 300
FONT_SIZE = 11
# ===================================================================

In [ ]:
import os, sys, json, subprocess
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from pathlib import Path

matplotlib.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "axes.titlesize": FONT_SIZE + 1,
    "xtick.labelsize": FONT_SIZE - 1,
    "ytick.labelsize": FONT_SIZE - 1,
    "legend.fontsize": FONT_SIZE - 1,
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "savefig.bbox": "tight",
})

WORK_DIR = Path("/kaggle/working")
OUTPUT_ROOT = WORK_DIR / "outputs"
CHECKPOINT_ROOT = WORK_DIR / "checkpoints"

REPO_URL = "https://github.com/PhoenixEvo/ct-slice-interpolation-3dgs.git"
REPO_DIR = WORK_DIR / "ct-slice-interpolation-3dgs"
if not (REPO_DIR / "src").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

fig_dir = OUTPUT_ROOT / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
prepared_dir = OUTPUT_ROOT / "prepared"

from src.utils.seed import set_seed
set_seed(SEED)

print(f"Figures will be saved to: {fig_dir}")

In [ ]:
# Helper: Generate predictions for all methods on a given case
from src.data.sparse_simulator import SparseSimulator
from src.models.classical_interp import ClassicalInterpolator
from src.evaluation.metrics import compute_psnr, compute_ssim

R = VIZ_SPARSE_RATIO
device = "cuda" if torch.cuda.is_available() else "cpu"

def get_all_predictions(case_idx, R):
    """Get predictions from all methods for a given case."""
    vol_path = prepared_dir / f"volume_case{case_idx}.npy"
    if not vol_path.exists():
        return None

    volume = np.load(vol_path)
    labels = None
    lab_path = prepared_dir / f"labels_case{case_idx}.npy"
    if lab_path.exists():
        labels = np.load(lab_path)

    sim = SparseSimulator(sparse_ratio=R)
    sparse = sim.simulate(volume)

    preds = {}

    # Classical
    classical = ClassicalInterpolator.interpolate_all_methods(
        observed_slices=sparse["observed_slices"],
        observed_indices=sparse["observed_indices"],
        target_indices=sparse["target_indices"],
    )
    preds.update(classical)

    # U-Net (load checkpoint if available)
    from src.models.unet2d import UNet2D
    from src.training.trainer_unet import TrainerUNet
    from src.utils.config import load_config

    config = load_config("configs/default.yaml")
    unet_ckpt_dir = CHECKPOINT_ROOT / f"unet_R{R}"
    unet_ckpt = unet_ckpt_dir / "best.pt"
    if not unet_ckpt.exists():
        unet_ckpt = unet_ckpt_dir / "final.pt"

    if unet_ckpt.exists():
        model = UNet2D(
            in_channels=config.get("unet", {}).get("in_channels", 2),
            out_channels=config.get("unet", {}).get("out_channels", 1),
            features=config.get("unet", {}).get("features", [32, 64, 128, 256]),
        )
        tr = TrainerUNet(model=model, train_loader=None, val_loader=None,
                         lr=1e-4, num_epochs=1, checkpoint_dir=str(unet_ckpt_dir), device=device)
        tr.load_checkpoint(unet_ckpt.name)

        unet_preds = []
        pairs = sim.get_interpolation_pairs(volume)
        for p in pairs:
            before = torch.from_numpy(p["slice_before"]).float()
            after = torch.from_numpy(p["slice_after"]).float()
            for _ in p["targets"]:
                pred = tr.predict_slice(before, after).squeeze(0).numpy()
                unet_preds.append(np.clip(pred, 0, 1))
        if unet_preds:
            preds["U-Net"] = np.stack(unet_preds, axis=-1)

    # 3DGS (load checkpoint if available)
    from src.training.trainer_3dgs import Trainer3DGS
    dgs_ckpt_dir = CHECKPOINT_ROOT / "3dgs" / f"case{case_idx}_R{R}"
    dgs_ckpt = dgs_ckpt_dir / "final.pt"

    if dgs_ckpt.exists():
        t = Trainer3DGS(
            volume=volume,
            observed_indices=sparse["observed_indices"],
            target_indices=sparse["target_indices"],
            config=config, labels=labels, device=device,
            checkpoint_dir=str(dgs_ckpt_dir),
        )
        t.load_checkpoint("final.pt")
        preds["3DGS (Ours)"] = t.render_interpolated_slices()

    return {
        "volume": volume,
        "labels": labels,
        "sparse": sparse,
        "preds": preds,
    }

print("Helper functions defined.")

In [ ]:
# ============= Figure 1 & 2: Slice Comparison + Error Maps =============

# Find available test cases
with open(prepared_dir / "split_info.json") as f:
    split_info = json.load(f)

available_viz_cases = [c for c in VIZ_CASES if (prepared_dir / f"volume_case{c}.npy").exists()]
if not available_viz_cases:
    available_viz_cases = [c for c in split_info["test"][:3]
                          if (prepared_dir / f"volume_case{c}.npy").exists()]

print(f"Visualizing cases: {available_viz_cases}")

for case_idx in available_viz_cases:
    data = get_all_predictions(case_idx, R)
    if data is None:
        continue

    sparse = data["sparse"]
    gt_targets = sparse["target_slices"]
    preds = data["preds"]

    # Select a representative target slice (middle of volume)
    n_targets = gt_targets.shape[2]
    target_i = n_targets // 2
    z_idx = int(sparse["target_indices"][target_i])
    gt = gt_targets[:, :, target_i]

    # Auto-detect zoom region near organ boundary
    H, W = gt.shape
    from scipy import ndimage
    edge = ndimage.sobel(gt)
    center_r, center_c = np.unravel_index(np.argmax(edge[H//4:3*H//4, W//4:3*W//4]),
                                          (H//2, W//2))
    center_r += H // 4
    center_c += W // 4
    zoom_size = min(H, W) // 4
    r1 = max(0, center_r - zoom_size // 2)
    r2 = min(H, r1 + zoom_size)
    c1 = max(0, center_c - zoom_size // 2)
    c2 = min(W, c1 + zoom_size)

    # Select methods to show (skip nearest for cleaner figure)
    show_methods = ["linear", "cubic"]
    if "U-Net" in preds:
        show_methods.append("U-Net")
    if "3DGS (Ours)" in preds:
        show_methods.append("3DGS (Ours)")

    ncols = len(show_methods) + 1

    # --- Figure 1: Slice comparison with zoom ---
    fig, axes = plt.subplots(2, ncols, figsize=(3.2 * ncols, 6.4))

    axes[0, 0].imshow(gt, cmap="gray", vmin=0, vmax=1)
    axes[0, 0].set_title("Ground Truth")
    axes[0, 0].axis("off")
    rect = plt.Rectangle((c1, r1), c2-c1, r2-r1, lw=2, ec="red", fc="none")
    axes[0, 0].add_patch(rect)
    axes[1, 0].imshow(gt[r1:r2, c1:c2], cmap="gray", vmin=0, vmax=1)
    axes[1, 0].set_title("GT (zoom)")
    axes[1, 0].axis("off")

    for j, method in enumerate(show_methods):
        pred_vol = preds.get(method)
        if pred_vol is None:
            continue
        pred = pred_vol[:, :, target_i] if pred_vol.ndim == 3 else pred_vol
        psnr = compute_psnr(pred, gt)
        ssim = compute_ssim(pred, gt)

        axes[0, j+1].imshow(pred, cmap="gray", vmin=0, vmax=1)
        axes[0, j+1].set_title(f"{method}\n{psnr:.1f}dB / {ssim:.3f}")
        axes[0, j+1].axis("off")
        rect = plt.Rectangle((c1, r1), c2-c1, r2-r1, lw=2, ec="red", fc="none")
        axes[0, j+1].add_patch(rect)

        axes[1, j+1].imshow(pred[r1:r2, c1:c2], cmap="gray", vmin=0, vmax=1)
        axes[1, j+1].set_title(f"{method} (zoom)")
        axes[1, j+1].axis("off")

    fig.suptitle(f"Case {case_idx}, z={z_idx}, R={R}", fontsize=FONT_SIZE+2)
    plt.tight_layout()
    fig.savefig(fig_dir / f"fig1_comparison_case{case_idx}_R{R}.pdf")
    fig.savefig(fig_dir / f"fig1_comparison_case{case_idx}_R{R}.png")
    plt.show()

    # --- Figure 2: Error maps ---
    fig2, axes2 = plt.subplots(1, len(show_methods), figsize=(3.5 * len(show_methods), 3.5))
    if len(show_methods) == 1:
        axes2 = [axes2]

    max_err = 0
    for method in show_methods:
        pred_vol = preds.get(method)
        if pred_vol is not None:
            pred = pred_vol[:, :, target_i] if pred_vol.ndim == 3 else pred_vol
            max_err = max(max_err, np.abs(pred - gt).max())

    for j, method in enumerate(show_methods):
        pred_vol = preds.get(method)
        if pred_vol is None:
            continue
        pred = pred_vol[:, :, target_i] if pred_vol.ndim == 3 else pred_vol
        err = np.abs(pred - gt)
        im = axes2[j].imshow(err, cmap="hot", vmin=0, vmax=max_err)
        axes2[j].set_title(f"{method}\nMAE={err.mean():.4f}")
        axes2[j].axis("off")

    fig2.colorbar(im, ax=axes2, shrink=0.8, label="Absolute Error")
    fig2.suptitle(f"Error Maps - Case {case_idx}, z={z_idx}, R={R}")
    plt.tight_layout()
    fig2.savefig(fig_dir / f"fig2_error_case{case_idx}_R{R}.pdf")
    fig2.savefig(fig_dir / f"fig2_error_case{case_idx}_R{R}.png")
    plt.show()

    del data
    torch.cuda.empty_cache()

print(f"Figures 1 & 2 saved to {fig_dir}")

In [ ]:
# ============= Figure 3: PSNR/SSIM Bar Charts =============

# Load all results
dfs = []
for csv_name in ["classical_baselines/summary.csv", "unet_baseline/summary.csv", "3dgs/summary.csv"]:
    p = OUTPUT_ROOT / csv_name
    if p.exists():
        dfs.append(pd.read_csv(p))

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)

    method_order = ["nearest", "linear", "cubic", "unet", "3dgs"]
    method_labels = ["Nearest", "Linear", "Cubic", "U-Net", "3DGS\n(Ours)"]
    colors = ["#95a5a6", "#3498db", "#2ecc71", "#e74c3c", "#9b59b6"]

    for R in sorted(df_all["sparse_ratio"].unique()):
        sub = df_all[df_all["sparse_ratio"] == R]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

        psnr_means, psnr_stds = [], []
        ssim_means, ssim_stds = [], []
        avail_labels, avail_colors = [], []

        for m, label, color in zip(method_order, method_labels, colors):
            vals = sub[sub["method"] == m]
            if len(vals) == 0:
                continue
            psnr_means.append(vals["mean_psnr"].mean())
            psnr_stds.append(vals["mean_psnr"].std())
            ssim_means.append(vals["mean_ssim"].mean())
            ssim_stds.append(vals["mean_ssim"].std())
            avail_labels.append(label)
            avail_colors.append(color)

        x = np.arange(len(avail_labels))

        ax1.bar(x, psnr_means, yerr=psnr_stds, color=avail_colors, capsize=4, edgecolor="black", linewidth=0.5)
        ax1.set_xticks(x)
        ax1.set_xticklabels(avail_labels)
        ax1.set_ylabel("PSNR (dB)")
        ax1.set_title(f"PSNR Comparison (R={R})")
        ax1.grid(axis="y", alpha=0.3)
        for i, v in enumerate(psnr_means):
            ax1.text(i, v + psnr_stds[i] + 0.3, f"{v:.1f}", ha="center", fontsize=9, fontweight="bold")

        ax2.bar(x, ssim_means, yerr=ssim_stds, color=avail_colors, capsize=4, edgecolor="black", linewidth=0.5)
        ax2.set_xticks(x)
        ax2.set_xticklabels(avail_labels)
        ax2.set_ylabel("SSIM")
        ax2.set_title(f"SSIM Comparison (R={R})")
        ax2.grid(axis="y", alpha=0.3)
        for i, v in enumerate(ssim_means):
            ax2.text(i, v + ssim_stds[i] + 0.003, f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

        plt.tight_layout()
        fig.savefig(fig_dir / f"fig3_bar_R{R}.pdf")
        fig.savefig(fig_dir / f"fig3_bar_R{R}.png")
        plt.show()

    print(f"Figure 3 saved.")
else:
    print("No results found. Run notebooks 02-04 first.")

In [ ]:
# ============= Figure 4: Training Convergence Curves =============

# Find a 3DGS training history
history_files = list((CHECKPOINT_ROOT / "3dgs").rglob("history.json")) if (CHECKPOINT_ROOT / "3dgs").exists() else []

if history_files:
    with open(history_files[0]) as f:
        history = json.load(f)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    iters = history.get("iteration", [])

    if "loss_total" in history:
        axes[0, 0].plot(iters, history["loss_total"], label="Total", linewidth=1.5)
    if "loss_rec" in history:
        axes[0, 0].plot(iters, history["loss_rec"], label="Reconstruction", linewidth=1.5)
    axes[0, 0].set_xlabel("Iteration")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].set_title("Training Loss")
    axes[0, 0].legend()
    axes[0, 0].set_yscale("log")
    axes[0, 0].grid(True, alpha=0.3)

    if "loss_smooth" in history:
        axes[0, 1].plot(iters, history["loss_smooth"], label="Smoothness", linewidth=1.5, color="tab:orange")
    if "loss_edge" in history:
        axes[0, 1].plot(iters, history["loss_edge"], label="Edge", linewidth=1.5, color="tab:green")
    axes[0, 1].set_xlabel("Iteration")
    axes[0, 1].set_ylabel("Loss")
    axes[0, 1].set_title("Regularization Losses")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    if "psnr_train" in history:
        axes[1, 0].plot(iters, history["psnr_train"], linewidth=1.5, color="tab:blue")
        axes[1, 0].set_xlabel("Iteration")
        axes[1, 0].set_ylabel("PSNR (dB)")
        axes[1, 0].set_title("Training PSNR")
        axes[1, 0].grid(True, alpha=0.3)

    if "num_gaussians" in history:
        axes[1, 1].plot(iters, history["num_gaussians"], linewidth=1.5, color="tab:red")
        axes[1, 1].set_xlabel("Iteration")
        axes[1, 1].set_ylabel("Count")
        axes[1, 1].set_title("Number of Gaussians")
        axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle("3DGS Training Progress", fontsize=FONT_SIZE + 2)
    plt.tight_layout()
    fig.savefig(fig_dir / "fig4_convergence.pdf")
    fig.savefig(fig_dir / "fig4_convergence.png")
    plt.show()
    print("Figure 4 saved.")
else:
    print("No 3DGS training history found.")

In [ ]:
# ============= Figure 5: Ablation Study Bar Chart =============

abl_csv = OUTPUT_ROOT / "ablation" / "ablation_results.csv"
if abl_csv.exists():
    df_abl = pd.read_csv(abl_csv)

    # Regularization ablation
    reg_variants = ["no_reg", "smooth_only", "edge_only", "full"]
    reg_labels = ["No Reg.", "+Smooth", "+Edge", "Full (Ours)"]
    reg_colors = ["#e74c3c", "#f39c12", "#3498db", "#2ecc71"]

    df_reg = df_abl[df_abl["variant"].isin(reg_variants)]

    if len(df_reg) > 0:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

        psnr_means, psnr_stds = [], []
        ssim_means, ssim_stds = [], []
        avail_labels, avail_colors = [], []

        for v, label, color in zip(reg_variants, reg_labels, reg_colors):
            vals = df_reg[df_reg["variant"] == v]
            if len(vals) == 0:
                continue
            psnr_means.append(vals["mean_psnr"].mean())
            psnr_stds.append(vals["mean_psnr"].std())
            ssim_means.append(vals["mean_ssim"].mean())
            ssim_stds.append(vals["mean_ssim"].std())
            avail_labels.append(label)
            avail_colors.append(color)

        x = np.arange(len(avail_labels))

        ax1.bar(x, psnr_means, yerr=psnr_stds, color=avail_colors, capsize=4,
                edgecolor="black", linewidth=0.5)
        ax1.set_xticks(x)
        ax1.set_xticklabels(avail_labels, rotation=15)
        ax1.set_ylabel("PSNR (dB)")
        ax1.set_title("PSNR - Regularization Ablation")
        ax1.grid(axis="y", alpha=0.3)
        for i, v in enumerate(psnr_means):
            ax1.text(i, v + psnr_stds[i] + 0.2, f"{v:.2f}", ha="center", fontsize=9, fontweight="bold")

        ax2.bar(x, ssim_means, yerr=ssim_stds, color=avail_colors, capsize=4,
                edgecolor="black", linewidth=0.5)
        ax2.set_xticks(x)
        ax2.set_xticklabels(avail_labels, rotation=15)
        ax2.set_ylabel("SSIM")
        ax2.set_title("SSIM - Regularization Ablation")
        ax2.grid(axis="y", alpha=0.3)
        for i, v in enumerate(ssim_means):
            ax2.text(i, v + ssim_stds[i] + 0.002, f"{v:.4f}", ha="center", fontsize=9, fontweight="bold")

        plt.tight_layout()
        fig.savefig(fig_dir / "fig5_ablation.pdf")
        fig.savefig(fig_dir / "fig5_ablation.png")
        plt.show()
        print("Figure 5 saved.")
else:
    print("No ablation results found. Run notebook 05 first.")

In [ ]:
# ============= Summary of all generated figures =============

print("\n" + "="*60)
print("  Generated Figures Summary")
print("="*60)

for f in sorted(fig_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name} ({size_kb:.0f} KB)")

print(f"\nAll figures saved to: {fig_dir}")
print("PDF versions are recommended for LaTeX paper.")